---
# Part 1: What is a Neuron?
A neuron takes inputs, multiplies by weights, adds bias, then applies activation function.

**Formula:** `output = activation(w1*x1 + w2*x2 + b)`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simple neuron
def neuron(inputs, weights, bias):
    total = sum(w*x for w, x in zip(weights, inputs)) + bias
    output = 1 / (1 + np.exp(-total))  # sigmoid activation
    return output

inputs  = [1.0, 2.0, 3.0]
weights = [0.5, -0.3, 0.8]
bias    = 0.1

result = neuron(inputs, weights, bias)
print('Neuron output:', round(result, 4))

---
# Part 2: Perceptron (Simplest Neural Network)

In [ ]:
# Perceptron — AND gate
class Perceptron:
    def __init__(self, lr=0.1, epochs=100):
        self.lr = lr
        self.epochs = epochs

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        self.b = 0
        for _ in range(self.epochs):
            for xi, yi in zip(X, y):
                pred = 1 if np.dot(xi, self.w) + self.b >= 0 else 0
                err  = yi - pred
                self.w += self.lr * err * xi
                self.b += self.lr * err

    def predict(self, X):
        return [1 if np.dot(xi, self.w) + self.b >= 0 else 0 for xi in X]

X = np.array([[0,0],[0,1],[1,0],[1,1]])
y = np.array([0, 0, 0, 1])  # AND gate

p = Perceptron()
p.fit(X, y)
print('AND gate predictions:', p.predict(X))
print('Actual:              ', y.tolist())

---
# Part 3: Activation Functions

In [ ]:
x = np.linspace(-5, 5, 100)

sigmoid = 1 / (1 + np.exp(-x))
relu    = np.maximum(0, x)
tanh    = np.tanh(x)

plt.figure(figsize=(12, 3))

plt.subplot(1,3,1)
plt.plot(x, sigmoid, color='blue')
plt.title('Sigmoid'); plt.grid(True, alpha=0.3)

plt.subplot(1,3,2)
plt.plot(x, relu, color='green')
plt.title('ReLU'); plt.grid(True, alpha=0.3)

plt.subplot(1,3,3)
plt.plot(x, tanh, color='red')
plt.title('Tanh'); plt.grid(True, alpha=0.3)

plt.suptitle('Activation Functions')
plt.tight_layout()
plt.show()

---
# Part 4: ANN from Scratch (with Backpropagation)
Backpropagation = calculate error → go backwards → update weights

In [ ]:
# Simple ANN — XOR problem
class SimpleANN:
    def __init__(self, lr=0.5):
        self.lr = lr
        np.random.seed(42)
        self.W1 = np.random.randn(2, 4)
        self.b1 = np.zeros((1, 4))
        self.W2 = np.random.randn(4, 1)
        self.b2 = np.zeros((1, 1))

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def forward(self, X):
        self.A1 = self.sigmoid(X @ self.W1 + self.b1)
        self.A2 = self.sigmoid(self.A1 @ self.W2 + self.b2)
        return self.A2

    def train(self, X, y, epochs=3000):
        losses = []
        for i in range(epochs):
            out = self.forward(X)
            loss = np.mean((out - y)**2)
            losses.append(loss)
            # Backprop
            d2 = (out - y) * out * (1 - out)
            d1 = (d2 @ self.W2.T) * self.A1 * (1 - self.A1)
            self.W2 -= self.lr * self.A1.T @ d2
            self.W1 -= self.lr * X.T @ d1
        return losses

X_xor = np.array([[0,0],[0,1],[1,0],[1,1]])
y_xor = np.array([[0],[1],[1],[0]])

ann = SimpleANN(lr=0.5)
losses = ann.train(X_xor, y_xor, epochs=3000)

preds = (ann.forward(X_xor) > 0.5).astype(int)
print('XOR Predictions:', preds.flatten().tolist())
print('Actual:         ', y_xor.flatten().tolist())

plt.plot(losses, color='purple')
plt.title('Training Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.grid(True, alpha=0.3); plt.show()

---
# Part 5: Data Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pandas as pd

# Sample data
np.random.seed(42)
df = pd.DataFrame({
    'age':    np.random.randint(20, 60, 100),
    'income': np.random.randint(30000, 100000, 100),
    'label':  np.random.randint(0, 2, 100)
})

print('Dataset shape:', df.shape)
print(df.head())

X = df[['age','income']].values
y = df['label'].values

# Normalize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('\nBefore scaling mean:', X.mean(axis=0).round(1))
print('After  scaling mean:', X_scaled.mean(axis=0).round(3))

# Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

---
# Part 6: Computer Vision — CNN
- **Conv2D** → finds patterns (edges, shapes)
- **MaxPooling** → shrinks image size
- **Flatten** → converts to 1D
- **Dense** → classifies

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load CIFAR-10
(X_tr, y_tr), (X_te, y_te) = keras.datasets.cifar10.load_data()
X_tr, X_te = X_tr / 255.0, X_te / 255.0

names = ['airplane','car','bird','cat','deer','dog','frog','horse','ship','truck']
print('Train shape:', X_tr.shape)

# Show sample images
plt.figure(figsize=(12, 3))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(X_tr[i])
    plt.title(names[y_tr[i][0]], fontsize=8)
    plt.axis('off')
plt.suptitle('CIFAR-10 Samples')
plt.tight_layout(); plt.show()

In [ ]:
# Build simple CNN
model_cnn = keras.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_cnn.summary()

history = model_cnn.fit(X_tr, y_tr, epochs=5, batch_size=64, validation_split=0.1, verbose=1)
_, acc = model_cnn.evaluate(X_te, y_te, verbose=0)
print(f'Test Accuracy: {acc*100:.2f}%')

---
# Part 7: Image Augmentation
Augmentation = artificially creates more training data by flipping, rotating, zooming images

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2
)

# Show augmented versions
sample = X_tr[:1]
plt.figure(figsize=(12, 3))
plt.subplot(1,6,1); plt.imshow(sample[0]); plt.title('Original'); plt.axis('off')

i = 2
for batch in datagen.flow(sample, batch_size=1):
    plt.subplot(1,6,i); plt.imshow(batch[0]); plt.title(f'Aug {i-1}'); plt.axis('off')
    i += 1
    if i > 6: break

plt.suptitle('Image Augmentation')
plt.tight_layout(); plt.show()

---
# Part 8: Transfer Learning
Use a pretrained model (already trained on millions of images), then add our own classification head.

In [ ]:
from tensorflow.keras.applications import MobileNetV2

# Load MobileNetV2 pretrained on ImageNet
base = MobileNetV2(input_shape=(96,96,3), include_top=False, weights='imagenet')
base.trainable = False  # freeze — don't change these weights

# Add our own layers on top
model_tl = keras.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_tl.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print('Trainable layers:', len([l for l in model_tl.layers if l.trainable]))
model_tl.summary()

# Resize images for MobileNet
X_tr_r = tf.image.resize(X_tr[:3000], (96,96)).numpy()
X_te_r = tf.image.resize(X_te[:500],  (96,96)).numpy()

history_tl = model_tl.fit(X_tr_r, y_tr[:3000], epochs=3, batch_size=32, validation_split=0.1, verbose=1)
_, tl_acc = model_tl.evaluate(X_te_r, y_te[:500], verbose=0)
print(f'Transfer Learning Accuracy: {tl_acc*100:.2f}%')

---
# Part 9: Fine-Tuning
After feature extraction, unfreeze the top layers of the base model and retrain at a very low learning rate.

In [ ]:
# Unfreeze top 20 layers of base model
base.trainable = True
for layer in base.layers[:-20]:
    layer.trainable = False

# Lower learning rate for fine-tuning
model_tl.compile(optimizer=keras.optimizers.Adam(1e-5),
                 loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history_ft = model_tl.fit(X_tr_r, y_tr[:3000], epochs=3, batch_size=32, validation_split=0.1, verbose=1)
_, ft_acc = model_tl.evaluate(X_te_r, y_te[:500], verbose=0)
print(f'After Fine-Tuning Accuracy: {ft_acc*100:.2f}%')

---
#MNIST Handwritten Digit Recognition
#Build ANN + CNN using Keras/TensorFlow on MNIST dataset

In [ ]:
# Load MNIST
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Normalize
X_train, X_test = X_train / 255.0, X_test / 255.0

print('Train:', X_train.shape, '| Test:', X_test.shape)

# Show sample digits
plt.figure(figsize=(12, 3))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(X_train[i], cmap='gray')
    plt.title(f'Label: {y_train[i]}')
    plt.axis('off')
plt.suptitle('MNIST Sample Digits')
plt.tight_layout(); plt.show()

In [ ]:
# ---- MODEL 1: ANN ----
ann = keras.Sequential([
    layers.Flatten(input_shape=(28,28)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='ANN')

ann.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
ann.summary()

h_ann = ann.fit(X_train, y_train, epochs=5, batch_size=128, validation_split=0.1, verbose=1)
_, ann_acc = ann.evaluate(X_test, y_test, verbose=0)
print(f'\nANN Test Accuracy: {ann_acc*100:.2f}%')

In [ ]:
# ---- MODEL 2: CNN ----
X_train_c = X_train.reshape(-1, 28, 28, 1)
X_test_c  = X_test.reshape(-1, 28, 28, 1)

cnn = keras.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
], name='CNN')

cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn.summary()

h_cnn = cnn.fit(X_train_c, y_train, epochs=5, batch_size=128, validation_split=0.1, verbose=1)
_, cnn_acc = cnn.evaluate(X_test_c, y_test, verbose=0)
print(f'\nCNN Test Accuracy: {cnn_acc*100:.2f}%')

In [ ]:
# ---- RESULTS ----
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(h_ann.history['accuracy'], label='ANN Train')
axes[0].plot(h_cnn.history['accuracy'], label='CNN Train')
axes[0].plot(h_ann.history['val_accuracy'], label='ANN Val', linestyle='--')
axes[0].plot(h_cnn.history['val_accuracy'], label='CNN Val', linestyle='--')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(h_ann.history['loss'], label='ANN')
axes[1].plot(h_cnn.history['loss'], label='CNN')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('MNIST Training Results')
plt.tight_layout(); plt.show()

# Confusion Matrix
y_pred = np.argmax(cnn.predict(X_test_c, verbose=0), axis=1)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix — CNN'); plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()

print('='*40)
print(f'ANN Accuracy: {ann_acc*100:.2f}%')
print(f'CNN Accuracy: {cnn_acc*100:.2f}%')
print('='*40)

In [ ]:
# Show predictions
preds = np.argmax(cnn.predict(X_test_c[:20], verbose=0), axis=1)

plt.figure(figsize=(14, 3))
for i in range(20):
    plt.subplot(2, 10, i+1)
    plt.imshow(X_test[i], cmap='gray')
    color = 'green' if preds[i] == y_test[i] else 'red'
    plt.title(f'P:{preds[i]}', color=color, fontsize=8)
    plt.axis('off')
plt.suptitle('Predictions (Green=Correct, Red=Wrong)')
plt.tight_layout(); plt.show()